# Run map and coverage

## Overview
Each routine run leaves an `output_info` file that lists the DS1 measurements it made, and its new values should then show up in DS2 (or, for benchmarking, in DS3). Everything later joins the sources through these runs, so we need to know the links point at the right rows, the values agree along the way, and where the links are missing.

**Question.** Can we trace each routine run from its `output_info` file to DS1, DS2 and DS3? And where is data missing, and why?

**Method.** We build the run map with `uv run build-run-map`, which writes three tables to `data/derived/run_map/`: `runs` (one row per run, with the DS2 snapshots around it and its DS3 run), `run_entities` (old, new and flags per run, entity and param) and `run_results` (the DS1 ids each run claims). We check the links to DS2 and DS3 (section 1) and to DS1 (section 2), then which entities a run actually attempted (section 3), the DC bias freeze (section 4) and whether values agree along the chain (section 5). Results are in [Phase 01, A3](../../docs/phases/01_eda.md#a3-run-map-and-coverage).

In [1]:
import numpy as np
import pandas as pd

from qpu_drift.derived.run_map import load_run_entities, load_run_map, load_run_results
from qpu_drift.sources.measurements import load_measurements

# Run map tables, plus the DS1 spine (id, type and start of every measurement)
runs = load_run_map()
ent = load_run_entities()
ids = load_run_results()
ds1 = load_measurements()[["id", "measurement_type", "time_start"]]

## 1. The DS2 and DS3 links pair the right runs

Each `output_info` file is one run, with its start taken from the file path. A run sits between two DS2 snapshots: `snap_before`, the last one at or before the start (old values), and `snap_after`, the first one after it (written values). Runs before DS2 starts have neither.

Benchmarking runs link to DS3 through `kpi_l2` `measured_at`, which comes from an 8 h backward as-of join ([datasheet 3, section 6](../../docs/datasheets/03_benchmarking_files.md#6-preprocessing-and-cleaning)), so the lag is bounded by construction. To check it pairs the right files, we compare the served $R^2$ with `fit_quality` in the linked file, and in the file before it as a control.

| Routine | Runs | DS1 | DS2 | DS3 |
|---|---|---|---|---|
| `calibrate_1q` | 1,298 | `result_ids` per qubit and step | Updates it | - |
| `calibrate_2q` | 2,007 | `result_ids` per coupling | Updates it | - |
| `mixer_calibration` | 655 | `result_ids` | Updates it | - |
| `benchmarking` | 663 | `result_ids` per group | - | Writes the JSON about 3 h after the start |

In [2]:
# Per routine: runs, date span, empty runs, and runs linked to DS2 and DS3
runs.groupby("routine").agg(
    runs=("start", "size"),
    first=("start", "min"),
    last=("start", "max"),
    no_result_ids=("n_result_ids", lambda s: int((s == 0).sum())),
    with_ds2=("snap_before", "count"),
    with_ds3=("ds3_created_at", "count"),
)

,runs,first,last,no_result_ids,with_ds2,with_ds3
routine,,,,,,
benchmarking,663,2024-04-15 03:14:36,2026-08-17 20:06:09,6,542,563
calibrate_1q,1298,2024-03-21 17:03:19,2026-08-18 08:45:39,20,1121,0
calibrate_2q,2007,2024-04-15 08:43:05,2026-08-18 09:03:54,25,1117,0
mixer_calibration,655,2024-04-11 23:02:22,2026-08-17 19:05:39,1,549,0


Every routine has files from 2024-03 or 2024-04 to 2026-08, and only a few are empty (no `result_ids`). The next cell checks the DS2 and DS3 links behind the last two columns, then looks at the DS3 runs and files that don't link, and at the empty files.

In [3]:
from qpu_drift.sources.benchmarking import load_runs_l2
from qpu_drift.sources.hardware_models import load_index

# DS2: snapshot timing and written values
r = runs.dropna(subset=["snap_before"])
before_min = (r["start"] - r["snap_before"]).dt.total_seconds() / 60
print(
    f"snap_before more than 5 min before the start: {int((before_min > 5).sum())} of "
    f"{len(r):,} runs (max {before_min.max():.1f} min)"
)
print("snap_after after the start, max:", (r["snap_after"] - r["start"]).max())

# DS2 snapshots that are not the snap_before of any run
snaps = pd.to_datetime(load_index()["timestamp"]).unique()
print(
    f"DS2 snapshots: {len(snaps):,}; "
    f"not the snap_before of any run: {len(set(snaps) - set(r['snap_before'])):,}"
)

# Changed values, and how many of them were written to DS2
e = ent.dropna(subset=["written"])
e = e[e["changed"].fillna(False)]
e1 = e[e["routine"] == "calibrate_1q"]
e2 = e[(e["routine"] == "calibrate_2q") & (e["param"] == "amp")]
miss = e2[~e2["written"].astype(bool)]
print(f"1q changed: {len(e1):,}, written: {int(e1['written'].sum()):,}")
print(
    f"2q amp changed: {len(e2):,}, "
    f"not written: {len(miss)}, runs: {miss['start'].unique().tolist()}"
)

# DS3: links and the DS3 runs with no benchmarking file
b = runs[runs["routine"] == "benchmarking"].sort_values("start")

# Benchmarking files linked to a DS3 run, and the link lag
linked = b.dropna(subset=["ds3_created_at"])
lag = linked["ds3_created_at"] - linked["start"]
ds3 = load_runs_l2()["created_at"].astype("datetime64[ns]")
print(
    f"\nDS3 runs: {len(ds3)}; linked: {linked['ds3_created_at'].nunique()} to {len(linked)} files; "
    f"lag min {lag.min()}, median {lag.median()}, max {lag.max()}"
)

# DS3 runs with no linked file, and the benchmarking files either side of each
un = ds3[~ds3.isin(linked["ds3_created_at"])].sort_values().to_frame()
un = pd.merge_asof(
    un,
    b[["start", "n_result_ids"]].rename(columns={"start": "prev_file"}),
    left_on="created_at",
    right_on="prev_file",
)
un = pd.merge_asof(
    un,
    b[["start"]].rename(columns={"start": "next_file"}),
    left_on="created_at",
    right_on="next_file",
    direction="forward",
)

# Where each unlinked DS3 run falls
un["where"] = "other"
un.loc[(un["next_file"] - un["prev_file"]) > pd.Timedelta("30D"), "where"] = (
    "in a gap of over 30 days between files"
)
un.loc[un["next_file"].isna(), "where"] = "after the last file"
display(
    un.groupby("where").agg(
        n=("created_at", "size"),
        first=("created_at", "min"),
        last=("created_at", "max"),
        prev_file=("prev_file", "max"),
        next_file=("next_file", "min"),
    )
)
display(un[un["where"] == "other"])

# Benchmarking files with no DS3 run
nods3 = b[b["ds3_created_at"].isna()]
first_link = linked["start"].min()

# Why each file has no DS3 run
group = np.select(
    [
        nods3["start"] < first_link,
        nods3["n_result_ids"] == 0,
        nods3["start"].dt.date.isin(set(linked["start"].dt.date)),
    ],
    ["before the first linked file", "empty", "on a day with a linked run"],
    "no DS3 run that day",
)
print(f"files with no DS3 run: {len(nods3)}", pd.Series(group).value_counts().to_dict())

# Days with no DS3 run, and streaks of 3 days or more
days = pd.Series(sorted(nods3.loc[group == "no DS3 run that day", "start"].dt.normalize().unique()))
print("days with no DS3 run:", len(days))
streak = days.groupby((days.diff() != pd.Timedelta("1D")).cumsum()).agg(["min", "max", "size"])
display(streak[streak["size"] >= 3])

# Empty runs
empty = runs[runs["n_result_ids"] == 0]
e1q = empty[empty["routine"] == "calibrate_1q"].sort_values("start")
e2q = empty[empty["routine"] == "calibrate_2q"].sort_values("start")

# Each empty 1q run with an empty 2q run within 10 min
pair = pd.merge_asof(
    e1q[["start"]],
    e2q[["start"]].rename(columns={"start": "start_2q"}),
    left_on="start",
    right_on="start_2q",
    direction="nearest",
    tolerance=pd.Timedelta("10min"),
)
gap_s = (pair["start_2q"] - pair["start"]).dt.total_seconds()
print(
    f"\nempty 1q runs with an empty 2q run within 10 min: {gap_s.notna().sum()} "
    f"of {len(e1q)} (median gap {gap_s.median():.0f} s)"
)

# New values and warnings on the empty 2q runs
x = ent.merge(e2q[["routine", "start"]], on=["routine", "start"])
print(
    f"empty 2q runs: {len(e2q)}; "
    f"entity rows with a new value: {int(x['new'].notna().sum())} of {len(x):,}; "
    f"n_warnings: {e2q['n_warnings'].value_counts(dropna=False).to_dict()}"
)

# DS1 CRUnitaryIZ rows in the hour after each empty 2q start
cr = ds1.loc[ds1["measurement_type"] == "CRUnitaryIZ", "time_start"]
n_cr = e2q["start"].map(lambda s: int(cr.between(s, s + pd.Timedelta("1h")).sum()))
print(
    "CRUnitaryIZ rows in the hour after each empty 2q start:",
    n_cr.value_counts().sort_index().to_dict(),
)

snap_before more than 5 min before the start: 1 of 3,329 runs (max 9.4 min)
snap_after after the start, max: 0 days 03:15:02
DS2 snapshots: 4,603; not the snap_before of any run: 1,274
1q changed: 98,825, written: 98,825
2q amp changed: 32,342, not written: 21, runs: [Timestamp('2025-05-12 09:04:18')]

DS3 runs: 655; linked: 563 to 563 files; lag min 0 days 01:25:48.156277, median 0 days 02:58:11.283408, max 0 days 06:42:51.739882


,n,first,last,prev_file,next_file
where,,,,,
after the last file,16,2026-08-19 22:08:30.334313,2026-09-14 22:01:36.723743,2026-08-17 20:06:09,NaT
in a gap of over 30 days between files,75,2024-05-24 20:16:42.918570,2024-08-17 02:50:01.260564,2024-05-18 18:10:05,2024-10-15 14:46:30
other,1,2026-08-10 13:56:30.836100,2026-08-10 13:56:30.836100,2026-08-10 13:53:57,2026-08-11 20:13:46


,created_at,prev_file,n_result_ids,next_file,where
75,2026-08-10 13:56:30.836100,2026-08-10 13:53:57,0,2026-08-11 20:13:46,other


files with no DS3 run: 100 {'before the first linked file': 57, 'no DS3 run that day': 38, 'on a day with a linked run': 3, 'empty': 2}
days with no DS3 run: 37


,min,max,size
2,2025-03-18,2025-03-20,3
8,2026-01-19,2026-01-25,7
21,2026-07-02,2026-07-05,4



empty 1q runs with an empty 2q run within 10 min: 14 of 20 (median gap 46 s)
empty 2q runs: 25; entity rows with a new value: 0 of 3,865; n_warnings: {np.int64(0): 22, <NA>: 3}
CRUnitaryIZ rows in the hour after each empty 2q start: {0: 23, 64: 1, 160: 1}


`snap_before` is within 5 min of the start for all but one of 3,329 runs (9.4 min), and `snap_after` comes at most 3 h 15 min after. 1,274 of the 4,603 DS2 snapshots aren't the `snap_before` of any run, so we link a run to the first snapshot after its start, not to the next run's. All 98,825 changed 1q values are written, and all but 21 of the 32,342 changed 2q amplitudes, all 21 in one run (2025-05-12 09:04).

`measured_at` links 563 of the 655 DS3 runs, each to its own benchmarking file, 1 h 26 min to 6 h 43 min after the start (median 2 h 58 min). Of the other 92, 75 fall in the gap between the files of 2024-05-18 and 2024-10-15, 16 come after the last file, and one comes three minutes after an empty file (2026-08-10).

The other way round, 100 benchmarking files have no DS3 run: 57 from before the first linked file, 2 empty, and 3 reruns on a day that has a linked run. That leaves 38 files on 37 days with no linked DS3 run that day, including streaks on 2025-03-18 to 03-20, 2026-01-19 to 01-25 and 2026-07-02 to 07-05.

Empty files are few: 25 2q, 20 1q, 6 benchmarking and 1 mixer. 14 of the 20 empty 1q runs have an empty 2q run within 10 min (median 46 s apart). The empty 2q runs carry no new values or warnings, and 23 of the 25 have no `CRUnitaryIZ` row in DS1 in the hour after.

Next, the share of non-empty runs linked to each source, by period.

In [4]:
# Valid DS1 ids per run (`valid`, section 2), then the share of runs linked to each source by period
n_valid = ids[ids["valid"]].groupby(["routine", "start"]).size().rename("n_valid")
cov = runs[runs["n_result_ids"] > 0].join(n_valid, on=["routine", "start"])
cov["period"] = pd.cut(
    cov["start"],
    pd.to_datetime(["2000-01-01", "2024-09-18", "2025-01-01", "2100-01-01"]),
    right=False,
    labels=["before 2024-09-18", "2024-09-18 to 2024-12-31", "from 2025-01-01"],
)
display(
    cov.groupby(["period", "routine"], observed=True)
    .agg(
        runs=("start", "size"),
        ds1=("n_valid", lambda s: s.notna().mean()),
        ds2=("snap_after", lambda s: s.notna().mean()),
        ds3=("ds3_created_at", lambda s: s.notna().mean()),
    )
    .round(3)
)
print(
    "first benchmarking run linked to DS3:", cov.loc[cov["ds3_created_at"].notna(), "start"].min()
)

runs  ds1  ds2    ds3
period                   routine                                 
before 2024-09-18        benchmarking         53  0.0  0.0  0.000
                         calibrate_1q         32  0.0  0.0  0.000
                         calibrate_2q         28  0.0  0.0  0.000
                         mixer_calibration    23  0.0  0.0  0.000
2024-09-18 to 2024-12-31 benchmarking         64  1.0  0.0  1.000
                         calibrate_1q        139  1.0  0.0  0.000
                         calibrate_2q        857  1.0  0.0  0.000
                         mixer_calibration    83  1.0  0.0  0.000
from 2025-01-01          benchmarking        540  1.0  1.0  0.924
                         calibrate_1q       1107  1.0  1.0  0.000
                         calibrate_2q       1097  1.0  1.0  0.000
                         mixer_calibration   548  1.0  1.0  0.000

first benchmarking run linked to DS3: 2024-10-15 14:46:30


Before 2024-09-18 no non-empty run links to anything. From then to the end of 2024, every run links to DS1 and every benchmarking run to DS3, but nothing links to DS2. From 2025 every run links to DS1 and DS2, and 92.4% of benchmarking runs link to DS3. The early DS1 gap is the restore (section 2).

Then the DS2 link the other way round: does each DS2 change belong to a run?

In [5]:
from qpu_drift.derived.run_map.clean import ds2_series
from qpu_drift.derived.run_map.constants import DS2_TEMPLATES
from qpu_drift.sources.hardware_models import load_values

# DS2 values of the tracked params
values = load_values()
series = ds2_series(values[values["path_template"].isin(DS2_TEMPLATES)])

# Changes after the first snapshot
changes = series[series["valid_from_ts"] > series["valid_from_ts"].min()]

# Snapshots that are the snap_after of a run, per routine
after = runs.dropna(subset=["snap_after"]).groupby("routine")["snap_after"].apply(set)

# Does each change land on a snap_after of the routine that owns it?
changes = changes.assign(
    owned=[t in after.get(r, set()) for r, t in zip(changes["routine"], changes["valid_from_ts"])]
)

# Share owned, per routine and param
changes.groupby(["routine", "param"])["owned"].agg(changes="size", owned="mean").round(3)

changes  owned
routine           param                          
calibrate_1q      amp                34337  0.994
                  drag               31945  0.995
                  freq               33144  0.993
calibrate_2q      amp                32585  0.992
                  cancel_scale_im    32585  0.992
                  cancel_scale_re    32585  0.992
                  cr_scale_im        32583  0.992
                  cr_scale_re        32583  0.992
mixer_calibration dc_bias_i          11775  0.992
                  dc_bias_q          10800  0.991
                  imbalance          27505  0.991
                  phase_offset       26571  0.991

99.1 to 99.5% of DS2 changes to the tracked params fall on the `snap_after` of a run of the routine that owns them. [A4](a4_windows_breaks.ipynb) (section 5) looks at the rest.

Next, whether the 38 files with no DS3 run differ from the linked ones. We compare the share of each file's `fit_quality` $R^2$ values below 0.9.

In [6]:
import json

from qpu_drift.paths import OUTPUT_INFO


def low_r2_share(file: str) -> float:
    """Share of a benchmarking file's fit_quality values below 0.9."""
    fq = json.loads((OUTPUT_INFO / file).read_text()).get("fit_quality") or {}
    v = [
        x
        for g in fq.values()
        if isinstance(g, dict)
        for x in g.values()
        if isinstance(x, (int, float))
    ]
    return float(np.mean(np.array(v) < 0.9)) if v else np.nan


# Non-empty benchmarking runs from the first DS3 link on
bench = runs[
    (runs["routine"] == "benchmarking")
    & (runs["start"] >= first_link)
    & (runs["n_result_ids"] > 0)
].copy()

# Group: linked to DS3, a rerun on a day that has a linked run, or no DS3 run that day
linked_days = set(bench.loc[bench["ds3_created_at"].notna(), "start"].dt.date)
bench["group"] = np.where(
    bench["ds3_created_at"].notna(),
    "linked",
    np.where(bench["start"].dt.date.isin(linked_days), "rerun", "no DS3"),
)

# Share of fit_quality values below 0.9 in each file
bench["low_r2"] = bench["file"].map(low_r2_share).round(3)

# Low-R2 share by group, then the no-DS3 files with more than half below 0.9
display(bench.groupby("group")["low_r2"].describe()[["count", "50%", "max"]].round(3))
bench.loc[
    (bench["group"] == "no DS3") & (bench["low_r2"] > 0.5),
    ["start", "n_result_ids", "n_warnings", "low_r2"],
]

,count,50%,max
group,,,
linked,562.0,0.104,1.000
no DS3,38.0,0.128,1.000
rerun,3.0,0.119,0.125


,start,n_result_ids,n_warnings,low_r2
3760,2026-03-09 20:05:42,69,127,1.000
3766,2026-03-10 19:58:05,46,73,1.000
4349,2026-07-02 20:00:36,65,86,0.772
4355,2026-07-03 19:54:50,29,41,0.768
4361,2026-07-04 19:50:08,44,51,0.761
4367,2026-07-05 19:48:42,44,52,0.761


Their fits look much like the linked ones: a median of 12.8% of $R^2$ values below 0.9, against 10.4%, and both groups reach 100%. Six of the 38 are over half: 2026-03-09 and 03-10 (100%) and 2026-07-02 to 07-05 (76 to 77%). Both stretches fall inside the chip-wide f01 shifts in [A2](a2_entity_ids.ipynb).

Last, the check that the DS3 link pairs the right files: served $R^2$ against `fit_quality` in the linked file and in the file before.

In [7]:
from qpu_drift.derived.run_map import differs
from qpu_drift.sources.benchmarking import load_kpi
from qpu_drift.sources.benchmarking.output_info import output_info_rows

# DS3 pairing: served R2 (Level 1, verbatim) against output_info fit_quality
# of the linked file and of the file before it
served = load_kpi().dropna(subset=["r2"])[["created_at", "metric", "element", "r2"]]
served["created_at"] = served["created_at"].astype("datetime64[ns]")

# Non-empty benchmarking files in order, each with the start of the file before it
bfiles = b[b["n_result_ids"] > 0].reset_index(drop=True)
bfiles["prev_start"] = bfiles["start"].shift(1)

# R2 of every KPI in each output_info file
oi_r2 = (
    pd.DataFrame(
        [
            row
            for start, file in zip(bfiles["start"], bfiles["file"])
            for row in output_info_rows(json.loads((OUTPUT_INFO / file).read_text()), start)
        ]
    )
    .dropna(subset=["r2"])
    .rename(columns={"measured_at": "file_start"})[["file_start", "metric", "element", "r2"]]
)

# Compare the served R2 with the linked file and, as a control, the file before
pairs = bfiles.dropna(subset=["ds3_created_at"]).astype({"ds3_created_at": "datetime64[ns]"})
res = []
for pairing, col in [("linked file", "start"), ("file before", "prev_start")]:
    p = (
        pairs[["ds3_created_at", col]]
        .rename(columns={"ds3_created_at": "created_at", col: "file_start"})
        .dropna()
    )

    # Match on run, file start, metric and element
    m = served.merge(p, on="created_at").merge(
        oi_r2, on=["file_start", "metric", "element"], suffixes=("_ds3", "_oi")
    )

    # Equal within the relative tolerance
    m["equal"] = ~differs(m["r2_ds3"], m["r2_oi"]).fillna(True).to_numpy(bool)
    res.append(m.assign(pairing=pairing))

# Per pairing: values compared, and runs where every value is equal
res = pd.concat(res, ignore_index=True)
print("served R2 compared from", res["created_at"].min(), "to", res["created_at"].max())
per_run = res.groupby(["pairing", "created_at"])["equal"].all()
res.groupby("pairing").agg(
    runs=("created_at", "nunique"), cells=("equal", "size"), equal=("equal", "mean")
).assign(runs_all_equal=per_run.groupby("pairing").sum()).round(4)

served R2 compared from 2025-07-07 23:10:10.082632 to 2026-08-16 21:53:56.049670


,runs,cells,equal,runs_all_equal
pairing,,,,
file before,338,41364,0.0,0
linked file,339,42094,1.0,339


Served $R^2$ equals `fit_quality` in the linked file for all 42,094 values (339 runs from 2025-07-07 on), and for none of the 41,364 in the file before. So each DS3 run is paired with the right benchmarking file.

## 2. DS1 links hold from the 2024-09-18 restore on

Each run lists the DS1 result ids it claims. We join each one to DS1 and call it `valid` only if the DS1 row starts within 24 h of the run - the same guard as the DS3 recovery ([datasheet 3, section 6](../../docs/datasheets/03_benchmarking_files.md#6-preprocessing-and-cleaning)). Then we see how much of DS1 the valid ids cover, by measurement type and quarter.

In [8]:
# DS1 restore date: its lowest id starts here (printed below)
RESTORE = pd.Timestamp("2024-09-18")

valid = ids[ids["valid"]]
print(f"claims: {len(ids):,}; distinct ids: {ids['result_id'].nunique():,}")

# 1q qubits per valid id, and valid ids claimed by more than one run
per_id = valid[valid["routine"] == "calibrate_1q"].groupby("result_id")["key"].nunique()
print(f"1q qubits per valid id: median {per_id.median():.0f}, max {per_id.max()}")
print(
    "valid ids claimed by more than one run:",
    int((valid.groupby("result_id")["start"].nunique() > 1).sum()),
)

# Claims that fail the 24 h check, and where they sit relative to the restore
bad = ids[~ids["valid"]]
print(
    f"\nnot valid: {len(bad):,} of {len(ids):,}; "
    f"after the restore: {int((bad['start'] >= RESTORE).sum())}"
)

# Days from run to DS1 row, for the failed claims from before the restore
gap = bad.merge(ds1, left_on="result_id", right_on="id")
pre = gap[gap["start"] < RESTORE]
print(
    f"runs before the restore: {int((runs['start'] < RESTORE).sum())}; "
    f"with a failed id: {len(pre[['routine', 'start']].drop_duplicates())}"
)
print(
    "days from run to DS1 row, before the restore:",
    ((pre["time_start"] - pre["start"]).dt.days).describe()[["min", "50%", "max"]].to_dict(),
)
print("lowest DS1 id:", ds1.loc[ds1["id"].idxmin(), ["id", "time_start"]].to_dict())

# The failed claim after the restore
display(
    gap.loc[
        gap["start"] >= RESTORE,
        ["routine", "start", "key", "step", "measurement_type", "time_start"],
    ]
)

# Share of DS1 rows claimed by a valid id: overall, by type and by quarter
ds1["claimed"] = ds1["id"].isin(set(valid["result_id"]))
print(f"DS1 rows claimed: {ds1['claimed'].mean():.1%} of {len(ds1):,}")

by_type = ds1.groupby("measurement_type")["claimed"].agg(rows="size", claimed="mean")
display(by_type.sort_values("rows", ascending=False).head(15).round(3))
print("types with no claimed row:", by_type.index[by_type["claimed"] == 0].tolist())
(
    ds1.groupby(ds1["time_start"].dt.to_period("Q"))["claimed"]
    .agg(rows="size", claimed="mean")
    .round(3)
)

claims: 739,230; distinct ids: 242,491
1q qubits per valid id: median 15, max 17
valid ids claimed by more than one run: 0

not valid: 18,472 of 739,230; after the restore: 1
runs before the restore: 148; with a failed id: 136
days from run to DS1 row, before the restore: {'min': 136.0, '50%': 153.0, 'max': 182.0}
lowest DS1 id: {'id': 1, 'time_start': Timestamp('2024-09-18 11:53:21.161967')}


,routine,start,key,step,measurement_type,time_start
18471,benchmarking,2025-08-22 08:26:52,C19-T11,rb2q,RandomizedBenchmarking2q,2025-07-10 21:48:41.528094


DS1 rows claimed: 53.7% of 444,080


,rows,claimed
measurement_type,,
CRUnitaryIZ,98812,0.598
MeasureBands,81249,0.994
LOLeakageOptimisation,81095,0.000
SidebandImbalanceOptimisation,40619,0.000
SidebandPhaseOptimisation,40618,0.000
MixerCalibration,40540,0.996
RandomizedBenchmarking2q,18487,0.997
XPi2Amp,9742,0.981
XPi2Drag,7565,0.978


types with no claimed row: ['CRDrag', 'CrLeakage', 'CrossResonanceRabi', 'DDropTemp', 'DDropTotal', 'FastQubitPeakFinder', 'FastResonatorPeakFinder', 'FastResonatorSpectroscopy', 'FineFrequencyTemp', 'InterleavedBenchmarking2q', 'LOLeakageOptimisation', 'QubitPeakFinder', 'Rabi12', 'RabiPopulation', 'RabiQubitTemperature', 'Ramsey12', 'RamseyJ', 'ReadoutAmp', 'ReadoutFrequency', 'ReadoutMatrix', 'ReadoutWidth', 'ResonatorChiMeasurement', 'ResonatorSpectroscopy', 'SidebandImbalanceOptimisation', 'SidebandPhaseOptimisation', 'XPi2Amp12', 'ZMapDDrop']


,rows,claimed
time_start,,
2024Q3,10341,0.413
2024Q4,66318,0.674
2025Q1,48793,0.496
2025Q2,59195,0.524
2025Q3,60281,0.521
2025Q4,60621,0.522
2026Q1,50974,0.517
2026Q2,57281,0.513
2026Q3,30276,0.513


The 739,230 claims hold 242,491 distinct ids, because a 1q id is listed under every qubit it measures (median 15, up to 17). No valid id is claimed by two runs.

18,472 claims fail the 24 h check. All but one come from 136 of the 148 runs before the DS1 restore on 2024-09-18 (DS1's lowest id, 1, starts then), and they point at rows 136 to 182 days later (median 153) - the ids were reused after the restore. [Datasheet 3](../../docs/datasheets/03_benchmarking_files.md#6-preprocessing-and-cleaning) sees the same for benchmarking ids. The one failure after the restore is an `rb2q` id from the 2025-08-22 benchmarking run, whose DS1 row starts on 2025-07-10.

Valid ids claim 53.7% of the 444,080 DS1 rows, and 49.6 to 52.4% a quarter from 2025. Most large types are 92 to 100% claimed. The exceptions are `CRUnitaryIZ` (60%) and the LO leakage and sideband sub-steps (0%), and 27 types are never claimed, readout types included.

## 3. A warning or an unchanged value isn't a rejection

We split each run into params per entity: freq, amp and drag per qubit (1q); amp, `cr_scale` and `cancel_scale` per coupling (2q, complex values split into `_re` and `_im`); and `dc_bias_i`, `dc_bias_q`, `imbalance` and `phase_offset` per channel (mixer). Each row gets three flags:

- **changed:** `new` differs from `old` in `hw_parameters`
- **warned:** named in `warnings` (NA when the file has none; mixer never writes them)
- **written:** the DS2 value differs between `snap_before` and `snap_after`

First we check that changed matches written, leaving out DC bias after DS2 stops recording it (2026-05-25 22:01). Then we ask which rows a run actually attempted, since a missing or unchanged value can only be a rejection if the run tried. For 1q, **in_run** means the qubit has an id at any step, and `measured` means an id at a step that writes the param (`STEPS_1Q`: Ramsey or FineFrequency for freq, XPi2Amp for amp, XPi2Drag for drag). For 2q, `measured` means the coupling has an id.

In [9]:
# Last DS2 DC-bias change; later DC-bias rows are left out below
BIAS_FROZEN = series.loc[series["param"].str.startswith("dc_bias"), "valid_from_ts"].max()
print("last DS2 DC-bias change:", BIAS_FROZEN)

# Changed against written, by warned state
d = ent.dropna(subset=["before", "after", "changed"])
frozen = d["param"].str.startswith("dc_bias") & (d["start"] > BIAS_FROZEN)
d = d[~frozen]
d.groupby(["routine", "param", "warned", "changed"], dropna=False)["written"].agg(
    rows="size", written="mean"
).round(4)

last DS2 DC-bias change: 2026-05-25 22:01:25


rows  written
routine           param           warned changed                
calibrate_1q      amp             False  False     1304      0.0
                                         True     23488      1.0
                                  True   False      434      0.0
                                         True     10646      1.0
                  drag            False  False     3120      0.0
                                         True     31378      1.0
                                  True   False      963      0.0
                                         True       411      1.0
                  freq            False  False     2969      0.0
                                         True     31051      1.0
                                  True   False        1      0.0
                                         True      1851      1.0
calibrate_2q      amp             False  True     32124   0.9993
                                  True   True       218      1.0
                  cancel_scale_im False  True     32124   0.9993
                                  True   True       218      1.0
                  cancel_scale_re False  True     32124   0.9993
                                  True   True       218      1.0
                  cr_scale_im     False  True     32124   0.9993
                                  True   True       218      1.0
                  cr_scale_re     False  True     32124   0.9993
                                  True   True       218      1.0
mixer_calibration dc_bias_i       <NA>   False    17980   0.0014
                                         True     11771   0.9902
                  dc_bias_q       <NA>   False    18952   0.0012
                                         True     10799   0.9893
                  imbalance       <NA>   False     7808   0.0003
                                         True     27328   0.9977
                  phase_offset    <NA>   False     9285   0.0002
                                         True     25851   0.9975

Changed values are written for 100% of 1q, all but 21 of the 2q and 98.9 to 99.8% of mixer values, and unchanged ones almost never are (under 0.2%). A warning doesn't stop the write: 10,646 warned amp rows are changed and written. A ladder warning (`... with N=`) names one failed step, and the earlier steps still apply.

Next, the 2q revert warnings and what they leave in DS2.

In [10]:
import re

from qpu_drift.sources.output_info import iter_files

# Revert warnings in calibrate_2q files: lowest N per run and coupling, and repeats
reverts = (
    pd.DataFrame(
        [
            (start, f"{m[1]}-{m[2]}", int(m[3]))
            for _, start, _, oi in iter_files(OUTPUT_INFO, routines=["calibrate_2q"])
            for w in map(str, oi.get("warnings") or [])
            if (m := re.search(r"Reverting calibration for \((\d+), (\d+)\) with N=(\d+)", w))
        ],
        columns=["start", "entity", "N"],
    )
    .groupby(["start", "entity"])["N"]
    .agg(N="min", reverts="size")
    .reset_index()
)
print("couplings reverted more than once in a run:", int((reverts["reverts"] > 1).sum()))

# What each revert left in the amp rows: new value and DS2 value
amp2q = ent[(ent["routine"] == "calibrate_2q") & (ent["param"] == "amp")]
rv = reverts.merge(amp2q, on=["start", "entity"], how="left", indicator=True)
print("reverts with no amp row:", int((rv["_merge"] == "left_only").sum()))

# Equal within the relative tolerance
close = lambda a, b: (a - b).abs() <= 1e-9 * b.abs()

# Outcome of each revert
rv["outcome"] = np.select(
    [
        rv["new"].isna() & rv["after"].isna(),
        rv["new"].isna() & close(rv["after"], rv["old"]),
        rv["new"].isna(),
        rv["after"].isna(),
        close(rv["after"], rv["new"]),
    ],
    [
        "no new, no DS2",
        "no new, DS2 keeps old",
        "no new, DS2 differs from old",
        "new, no DS2",
        "DS2 = new (earlier step)",
    ],
    "other",
)
print(
    "no new, no DS2:",
    rv.loc[rv["outcome"] == "no new, no DS2", "start"].agg(["min", "max"]).to_dict(),
)

# Outcome by lowest N
pd.crosstab(rv["N"], rv["outcome"], margins=True)

couplings reverted more than once in a run: 853
reverts with no amp row: 0
no new, no DS2: {'min': Timestamp('2024-10-31 08:22:06'), 'max': Timestamp('2024-12-31 20:10:35')}


outcome,DS2 = new (earlier step),"new, no DS2","no new, DS2 differs from old","no new, DS2 keeps old","no new, no DS2",All
N,,,,,,
1,60,2,1,738,116,917
2,156,5,0,0,0,161
All,216,7,1,738,116,1078


1,078 run and coupling pairs have a `Reverting calibration ... with N=` warning (917 at lowest $N=1$, 161 at $N=2$), and 853 of them revert more than once. At $N=2$ almost all (156) leave DS2 at the `output_info` new value from an earlier step. At $N=1$, 738 have no new value and DS2 keeps the old one, and 116 have neither a new value nor a DS2 value (2024-10-31 to 12-31, before any DS2 link). That leaves 60 at $N=1$ where DS2 takes the new value, 7 with a new value but no DS2 value, and 1 where DS2 differs from old.

Then the DRAG warnings, and the amp rows with only a Rabi id.

In [11]:
# DRAG warnings: the ladder (with N=) and the pre-check (N = 0 here)
drag_w = pd.DataFrame(
    [
        (start, m[1] or m[3], int(m[2] or 0))
        for _, start, _, oi in iter_files(OUTPUT_INFO, routines=["calibrate_1q"])
        for w in map(str, oi.get("warnings") or [])
        if (
            m := re.search(
                r"DRAG calibration out of spec for Q(\d+) with N=(\d+)|Q(\d+) pulse DRAG out of spec",
                w,
            )
        )
    ],
    columns=["start", "entity", "N"],
)

# DRAG rows, with the lowest ladder N warned for each
drag = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "drag")]
ladder = drag_w[drag_w["N"] > 0].groupby(["start", "entity"])["N"].min().reset_index()
dm = drag.merge(ladder, on=["start", "entity"])
print("DRAG rows by lowest warned N and changed:")
display(pd.crosstab(dm["N"], dm["changed"]))

# Warned at N=2 but unchanged: how many were already 0
n2 = dm[(dm["N"] == 2) & dm["changed"].eq(False).fillna(False)]
print(f"warned at N=2 and unchanged: {len(n2)}; old already 0: {int((n2['old'] == 0).sum())}")

# Pre-check rows (N = 0) and what they leave as the new value
pc = drag.merge(
    drag_w.loc[drag_w["N"] == 0, ["start", "entity"]].drop_duplicates(), on=["start", "entity"]
)
print(
    f"pre-check rows: {len(pc)}; "
    f"new = 0: {int((pc['new'] == 0).sum())}; new != 0: {int((pc['new'] != 0).sum())}"
)

# Rabi-only amp: unchanged and unwarned, a Rabi id and no amp-ladder id
amp = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "amp")]

# Unchanged, unwarned amp rows
q = amp[amp["changed"].eq(False).fillna(False) & amp["warned"].eq(False).fillna(False)]

# Steps each qubit has ids for, per run
steps = (
    ids[ids["routine"] == "calibrate_1q"]
    .assign(entity=lambda d: d["key"].str.lstrip("Q"))
    .groupby(["start", "entity"])["step"]
    .agg(set)
)
q = q.join(steps, on=["start", "entity"])

# Rows with a Rabi id and no amp-ladder id
rabi_only = q["step"].map(lambda s: isinstance(s, set) and "rabi" in s and "amp" not in s)
print(
    f"\nunchanged, unwarned amp rows with only Rabi ids: {int(rabi_only.sum())} in "
    f"{q.loc[rabi_only, 'start'].nunique()} runs; "
    f"of them measured: {int(q.loc[rabi_only, 'measured'].fillna(False).sum())}"
)
print(q.loc[rabi_only, "start"].dt.to_period("Q").value_counts().to_dict())

DRAG rows by lowest warned N and changed:


changed,False,True
N,,
1,56,7
2,910,6
3,0,125
4,0,237


warned at N=2 and unchanged: 910; old already 0: 717
pre-check rows: 128; new = 0: 10; new != 0: 118

unchanged, unwarned amp rows with only Rabi ids: 589 in 37 runs; of them measured: 0
{Period('2025Q3', 'Q-DEC'): 465, Period('2026Q3', 'Q-DEC'): 79, Period('2025Q2', 'Q-DEC'): 30, Period('2026Q2', 'Q-DEC'): 15}


By lowest warned $N$, DRAG rows are mostly unchanged at $N=1$ (56 of 63) and $N=2$ (910 of 916), and always changed at $N=3$ (125) and $N=4$ (237). 717 of the 910 unchanged at $N=2$ were already 0. The pre-check warning (`pulse DRAG out of spec - resetting to zero`) names 128 rows, but only 10 end up at 0. Separately, 589 unchanged, unwarned amp rows have a Rabi id and no amp-ladder id (37 runs, 465 rows in 2025 Q3), and none of them is `measured`.

Next, which rows each run actually attempted.

In [12]:
# Attempted vs rejected: which rows the run left any trace for
state = lambda d: np.select(
    [d["new"].isna(), d["changed"].eq(True).fillna(False)], ["no new", "changed"], "unchanged"
)

# 1q: in the run = the qubit has an id at any step; measured = an id at
# the param's own steps
c1 = ids[ids["routine"] == "calibrate_1q"]
in_run = set(zip(c1["start"], c1["key"].str.lstrip("Q")))
r1 = ent[ent["routine"] == "calibrate_1q"].copy()
r1["in_run"] = [k in in_run for k in zip(r1["start"], r1["entity"])]
r1["state"] = state(r1)
print("calibrate_1q rows by in_run, measured and warned:")
display(
    pd.crosstab([r1["param"], r1["in_run"], r1["measured"], r1["warned"].astype(str)], r1["state"])
)

# 1q rows in the run with no step id, unwarned and unchanged
s1 = r1[
    r1["in_run"]
    & ~r1["measured"].fillna(False)
    & r1["warned"].eq(False).fillna(False)
    & (r1["state"] == "unchanged")
]
print("1q in the run, no step id, unwarned, unchanged: rows and runs by quarter")
display(
    s1.groupby([s1["start"].dt.to_period("Q"), "param"])
    .agg(rows=("entity", "size"), runs=("start", "nunique"))
    .unstack("param", fill_value=0)
)

# 2q: measured = the coupling has an id (written only on accept);
# warned = a revert or amplitude warning
r2 = ent[(ent["routine"] == "calibrate_2q") & (ent["param"] == "amp")].copy()

# Flag the rows of empty 2q runs
r2["empty_run"] = r2["start"].isin(
    set(runs.loc[(runs["routine"] == "calibrate_2q") & (runs["n_result_ids"] == 0), "start"])
)
r2["state"] = state(r2)
print("\ncalibrate_2q amp rows by empty run, measured and warned:")
display(pd.crosstab([r2["empty_run"], r2["measured"], r2["warned"].astype(str)], r2["state"]))

# 2q rows with no id, no warning and no new value, in non-empty runs
s2 = r2[
    ~r2["empty_run"]
    & ~r2["measured"].fillna(False)
    & r2["warned"].eq(False).fillna(False)
    & (r2["state"] == "no new")
]

# Non-empty runs per quarter, for scale
n_runs = r2.loc[~r2["empty_run"]].groupby(r2["start"].dt.to_period("Q"))["start"].nunique()
print("2q no id, no warning, no new value (non-empty runs): by quarter")
display(
    s2.groupby(s2["start"].dt.to_period("Q"))
    .agg(rows=("entity", "size"), runs=("start", "nunique"), couplings=("entity", "nunique"))
    .join(n_runs.rename("all_runs"))
)

# Spread over runs and over couplings
print("per run:", s2.groupby("start").size().describe()[["50%", "max"]].to_dict())
share = (
    s2.groupby("entity")["start"].nunique()
    / r2.loc[~r2["empty_run"]].groupby("entity")["start"].nunique()
)
print("couplings by share of their runs with no trace (top 10):")
share.dropna().sort_values(ascending=False).head(10).round(3)

calibrate_1q rows by in_run, measured and warned:


state                         changed  unchanged
param in_run measured warned                    
amp   False  False    False         0        816
                      True          3          0
      True   False    False         0        648
                      True         20        618
             True     False     27288          0
                      True      11951          0
drag  False  False    False         0        819
      True   False    False         0       2972
                      True         10        966
             True     False     36093          0
                      True        484          0
freq  False  False    False         0        819
      True   False    False         0       2786
                      True          0          1
             True     False     35744         12
                      True       1982          0

1q in the run, no step id, unwarned, unchanged: rows and runs by quarter


rows           runs          
param   amp drag freq  amp drag freq
start                               
2024Q2    0   95   71    0   27   23
2024Q3    0    6    6    0    2    2
2024Q4    0  410  411    0  137  137
2025Q1    0  364  365    0  147  147
2025Q2   50  368  352    6  176  176
2025Q3  465  364  364   31  182  182
2025Q4    0  362  362    0  181  181
2026Q1    1  295  295    1  149  149
2026Q2   47  426  368    2  176  176
2026Q3   85  282  192    9   96   96


calibrate_2q amp rows by empty run, measured and warned:


state                      changed  no new  unchanged
empty_run measured warned                            
False     False    False         0    2549          0
                   True          0     855          0
          True     False     36206       0      22693
                   True        225       0          0
True      False    False         0     704          0

2q no id, no warning, no new value (non-empty runs): by quarter


,rows,runs,couplings,all_runs
start,,,,
2024Q4,526,469,4,857
2025Q1,269,144,8,144
2025Q2,83,45,3,177
2025Q3,106,106,1,182
2026Q1,159,12,31,150
2026Q2,734,173,8,173
2026Q3,672,91,31,91


per run: {'50%': 1.0, 'max': 28.0}
couplings by share of their runs with no trace (top 10):


entity
24-23    0.330
19-11    0.181
2-9      0.140
26-35    0.138
22-18    0.107
10-11    0.107
18-14    0.055
13-14    0.055
23-22    0.054
10-1     0.026
Name: start, dtype: float64

Measured 1q amp and drag always change, and measured freq is unchanged in just 12 unwarned rows. Rows with no id for the qubit at all (816 amp, 819 drag, 819 freq) have no warning and keep their value, apart from 3 amp rows. Rows in the run with no step id are mostly unwarned and unchanged too: 2,972 drag, 2,786 freq and 648 amp.

For 2q, couplings with an id have 36,206 unwarned and 225 warned changed rows, plus 22,693 unwarned rows accepted unchanged. Couplings with no id never get a new value: 855 warned and 2,549 unwarned, plus 704 in empty runs. The 2,549 are a median of 1 per run (max 28). They show up in every non-empty run of 2025 Q1 (144), 2026 Q2 (173) and 2026 Q3 (91), and in 469 of 857 runs in 2024 Q4. Coupling 24-23 is hit in 33.0% of its runs, then 19-11 (18.1%), 2-9 (14.0%) and 26-35 (13.8%).

Last, what explains the 1q rows in the run with no step id.

In [13]:
# The open 1q rows: which qubits, and which steps the qubit has ids for in that run
o = s1.join(steps, on=["start", "entity"])
has = lambda step: o["step"].map(lambda s: step in s)

# Drag and freq rows on a qubit with a freq_shift step in that run
fd = o["param"].isin(["drag", "freq"])
fs = fd & has("freq_shift")
print(f"drag and freq rows: {int(fd.sum()):,}; with a freq_shift step: {int(fs.sum()):,}")
display(
    o[fs]
    .groupby("entity")
    .agg(rows=("start", "size"), first=("start", "min"), last=("start", "max"))
)

# Amp rows with a Rabi id and no amp id (the Rabi-only rows above)
am = o["param"] == "amp"
rabi = am & has("rabi") & ~has("amp")

print(f"amp rows: {int(am.sum())}; Rabi and no amp id: {int(rabi.sum())}")

# What neither explains
left = (fd & ~fs) | (am & ~rabi)
print(
    f"left unexplained: {int((fd & ~fs).sum())} drag and freq, "
    f"{int((am & ~rabi).sum())} amp, {int(left.sum())} in all"
)

drag and freq rows: 5,758; with a freq_shift step: 5,487


,rows,first,last
entity,,,
13,2526,2024-04-23 01:33:36,2026-08-18 08:45:39
2,2531,2024-04-23 01:33:36,2026-08-18 08:45:39
23,428,2024-04-23 01:33:36,2025-02-04 19:45:23
35,2,2024-05-08 23:15:05,2024-05-08 23:15:05


amp rows: 648; Rabi and no amp id: 589
left unexplained: 271 drag and freq, 59 amp, 330 in all


These rows aren't attempts. 5,487 of the 5,758 drag and freq rows are on qubits that had a `freq_shift` step in that run and no step of their own: Q2 (2,531) and Q13 (2,526) throughout, Q23 (428) up to 2025-02-04, and Q35 (2). 589 of the 648 amp rows are the Rabi-only rows above. That leaves 330 rows we can't explain here (271 drag and freq, 59 amp).

## 4. DS2 stops recording DC bias after 2026-05-25

DS2 stops recording DC bias after 2026-05-25 22:01 (section 3). If `output_info` keeps calibrating it, `old` should match the DS2 `before` value up to the freeze and stop matching after, so we check that by month.

In [14]:
# DC bias by month from 2026-01: does old match the DS2 value before the run,
# and is the new value written?
bias = ent[ent["param"].str.startswith("dc_bias")].dropna(subset=["before", "after", "old"])
bias.groupby(bias["start"].dt.to_period("M")).agg(
    runs=("start", "nunique"),
    old_is_before=("before", lambda s: (s == bias.loc[s.index, "old"]).mean()),
    written=("written", "mean"),
).round(3).loc["2026-01":]

,runs,old_is_before,written
start,,,
2026-01,31,1.000,0.355
2026-02,27,0.944,0.339
2026-03,18,0.943,0.474
2026-04,30,1.000,0.406
2026-05,27,0.713,0.288
2026-06,30,0.000,0.0
2026-07,32,0.000,0.0
2026-08,16,0.000,0.0


`old` matches the DS2 value in 94 to 100% of rows from 2026-01 to 2026-04 and in 71% in 2026-05, then in none from 2026-06 on, and nothing is written after the freeze. So `output_info` keeps calibrating DC bias, and after 2026-05-25 we take it from there, not from DS2.

## 5. Values agree along the chain

Sections 1 to 3 show the links exist; here we check the linked values agree. We call two values $a$ and $b$ equal when

$$|a - b| \le 10^{-9}\,|b|,$$

the run map's relative tolerance (`differs`), and test three links:

- **`output_info` and DS2.** `old` against `before`, and changed `new` against `after`, leaving out DC bias after the freeze.
- **DS1 and `output_info`.** The Ramsey fits in the local payloads (`data/raw/raw_ramsey/`) against the run's freq `old` and `new`, joined on the valid Ramsey id and qubit, split by whether the run also has a FineFrequency (`freq`) step for that qubit.
- **DS2 and DS3.** The `q{n}_drive` and `r{n}_measure` frame frequencies in the DS3 JSON (from 2026-05-21) against DS2 as of the benchmarking start and as of `created_at`. [Datasheet 3](../../docs/datasheets/03_benchmarking_files.md#4-composition-and-structure) says the frames copy the hardware model; this checks it.

The DS1 to DS3 link for benchmarking values is the datasheet 3 recovery, which has its own guards, so we don't repeat it.

In [15]:
from qpu_drift.derived.run_map import differs

# output_info against DS2
d = ent.dropna(subset=["old", "before"])

# Leave out DC bias after the freeze
d = d[~(d["param"].str.startswith("dc_bias") & (d["start"] > BIAS_FROZEN))]

# Changed rows that have a DS2 value after the run
c = d[d["changed"].fillna(False) & d["after"].notna()]

# Share equal: old against before, and new against after
pd.concat(
    {
        "old = before": d.assign(eq=~differs(d["old"], d["before"]).to_numpy(bool))
        .groupby(["routine", "param"])["eq"]
        .agg(["size", "mean"]),
        "new = after": c.assign(eq=~differs(c["new"], c["after"]).to_numpy(bool))
        .groupby(["routine", "param"])["eq"]
        .agg(["size", "mean"]),
    },
    axis=1,
).round(4)

old = before         new = after        
                                          size    mean        size    mean
routine           param                                                   
calibrate_1q      amp                    35872  1.0000       34134  1.0000
                  drag                   35872  1.0000       31789  1.0000
                  freq                   35872  1.0000       32902  1.0000
calibrate_2q      amp                    35744  1.0000       32342  0.9994
                  cancel_scale_im        35744  1.0000       32342  0.9994
                  cancel_scale_re        35744  1.0000       32342  0.9994
                  cr_scale_im            35744  1.0000       32342  0.9994
                  cr_scale_re            35744  1.0000       32342  0.9994
mixer_calibration dc_bias_i              29760  0.9885       11771  0.9908
                  dc_bias_q              29760  0.9883       10799  0.9894
                  imbalance              35136  1.0000       27328  0.9977
                  phase_offset           35136  1.0000       25851  0.9975

`old` equals `before` for all 1q, 2q, imbalance and phase values, and 98.8 to 98.9% of DC bias. Changed `new` equals `after` for 100% of 1q, 99.94% of 2q, 99.75 to 99.77% of imbalance and phase, and 98.9 to 99.1% of DC bias.

Next, the Ramsey fits in DS1 against the run's freq values.

In [16]:
from qpu_drift import decode
from qpu_drift.paths import RAW, RAW_RAMSEY

# DS1 Ramsey fits against output_info freq
root = RAW_RAMSEY

# Ramsey rows whose payload is on disk
ram = load_measurements(measurement_type="Ramsey")
ram["file"] = ram["relative_filepath"].map(lambda p: root / p.removeprefix("results/"))
local = ram[ram["file"].map(lambda p: p.exists())]
print(f"Ramsey rows: {len(ram):,}; payloads on disk: {len(local):,}")

# The fits inside each payload: target, old and new value
fits = pd.DataFrame(
    [
        (rid, x.target, x.old, x.new)
        for rid, path in zip(local["id"], local["file"])
        for x in decode.extract(decode.payload(path))
    ],
    columns=["result_id", "key", "ds1_old", "ds1_new"],
)

# Ramsey result id per run and qubit, and the qubits with a FineFrequency step
steps_1q = ids[ids["valid"] & (ids["routine"] == "calibrate_1q")]
link = steps_1q.loc[steps_1q["step"] == "ramsey", ["start", "key", "result_id"]]
print(
    "Ramsey ids per run and qubit:", link.groupby(["start", "key"]).size().value_counts().to_dict()
)
fine = (
    steps_1q.loc[steps_1q["step"] == "freq", ["start", "key"]].drop_duplicates().assign(fine=True)
)

# Chain: DS1 fit, run and qubit, output_info freq row, FineFrequency flag
freq = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "freq")].assign(
    key=lambda d: "Q" + d["entity"]
)
chain = (
    fits.merge(link, on=["result_id", "key"])
    .merge(freq[["start", "key", "old", "new"]], on=["start", "key"])
    .merge(fine, on=["start", "key"], how="left")
    .fillna({"fine": False})
)
print(
    f"fits: {len(fits):,}; "
    f"linked to a run and qubit: {len(chain):,}; with no new value: {int(chain['new'].isna().sum())}"
)

# Equal within the relative tolerance, split by whether a FineFrequency step is present
chain["old_equal"] = ~differs(chain["ds1_old"], chain["old"]).to_numpy(bool)
chain["new_equal"] = ~differs(chain["ds1_new"], chain["new"]).to_numpy(bool)
display(chain.groupby("fine")[["old_equal", "new_equal"]].agg(["size", "mean"]).round(4))

# Size of the new-value gap when a FineFrequency step is present
f = chain[chain["fine"]]
print(
    "output_info new - Ramsey new (kHz), with a FineFrequency step:",
    ((f["new"] - f["ds1_new"]) / 1e3)
    .describe()[["25%", "50%", "75%", "min", "max"]]
    .round(1)
    .to_dict(),
)

Ramsey rows: 2,525; payloads on disk: 2,501
Ramsey ids per run and qubit: {1: 36918}
fits: 36,977; linked to a run and qubit: 36,918; with no new value: 0


old_equal      new_equal        
           size mean      size    mean
fine                                  
False       983  1.0       983  1.0000
True      35935  1.0     35935  0.0003

output_info new - Ramsey new (kHz), with a FineFrequency step: {'25%': -8.3, '50%': 2.6, '75%': 13.4, 'min': -499.8, 'max': 581.2}


2,501 of the 2,525 Ramsey payloads are on disk, holding 36,977 fits, and 36,918 link to a run and qubit (one Ramsey id each). Ramsey `old` equals `output_info` `old` for all of them. Ramsey `new` matches all 983 fits without a FineFrequency step, but only 0.03% of the 35,935 with one. There, `output_info` new minus Ramsey new has a median of 2.6 kHz (IQR −8.3 to 13.4, range −499.8 to 581.2 kHz). So Ramsey `new` is the applied f01 only when the run has no FineFrequency step for the qubit.

Last, the DS3 frame frequencies against DS2.

In [17]:
# DS3 openpulse frames against DS2

# Drive and measure frame frequencies in each DS3 benchmarking JSON
rows = []
for path in sorted(
    (RAW / "benchmarking" / "benchmarking_json").glob("*.json")
):
    frames = (
        json.loads(path.read_text())
        .get("openpulse_features", {})
        .get("open_pulse", {})
        .get("frames", {})
    )
    for name, fr in frames.items():
        if m := re.fullmatch(r"[qr](\d+)_(drive|measure)", name):
            rows.append((path.stem, int(m[1]), m[2], fr["frequency"]))

# One row per frame, with the run time taken from the file name
frames = pd.DataFrame(rows, columns=["stem", "qubit", "channel", "ds3"])
frames["created_at"] = pd.to_datetime(frames["stem"], format="%Y-%m-%dT%H%M%S_%f").astype(
    "datetime64[ns]"
)

# Only runs up to the last DS2 snapshot can be compared
last_snap = snaps.max()
print(
    f"DS3 runs with frames: {frames['created_at'].nunique()} ({frames['created_at'].min():%Y-%m-%d} to "
    f"{frames['created_at'].max():%Y-%m-%d}); "
    f"up to the last DS2 snapshot ({last_snap:%Y-%m-%d %H:%M}): {frames.loc[frames['created_at'] <= last_snap, 'created_at'].nunique()}"
)
frames = frames[frames["created_at"] <= last_snap]

# DS2 drive and measure frame frequencies over time
pcf = values[values["path_template"].isin([
    "model.pulse_channels{Ch*.Q*.drive}.pulse_channel.frequency",
    "model.pulse_channels{Ch*.R*.measure}.pulse_channel.frequency",
])]
pcf = pcf[["valid_from_ts", "value_float"]].join(
    pcf["path"].str.extract(r"\{Ch\d+\.[QR](?P<qubit>\d+)\.(?P<channel>drive|measure)\}")
)
pcf = pcf.dropna(subset=["channel"]).astype({"qubit": int}).sort_values("valid_from_ts")

# Benchmarking file start of each linked DS3 run
bench_start = (
    runs.loc[runs["routine"] == "benchmarking", ["start", "ds3_created_at"]]
    .dropna()
    .astype({"ds3_created_at": "datetime64[ns]"})
)
frames = frames.merge(bench_start, left_on="created_at", right_on="ds3_created_at", how="left")

# DS2 value as of the benchmarking start, and as of created_at
out = []
for at in ["start", "created_at"]:
    j = pd.merge_asof(
        frames.dropna(subset=[at]).sort_values(at),
        pcf.rename(columns={"valid_from_ts": at}),
        on=at,
        by=["qubit", "channel"],
    )
    out.append(j.assign(at=at, equal=~differs(j["ds3"], j["value_float"]).to_numpy(bool)))
out = pd.concat(out)
out.groupby(["at", "channel"]).agg(
    runs=("created_at", "nunique"), frames=("equal", "size"), equal=("equal", "mean")
)

DS3 runs with frames: 85 (2026-05-21 to 2026-09-14); up to the last DS2 snapshot (2026-08-18 09:13): 69


runs  frames  equal
at         channel                     
created_at drive      69    2208    1.0
           measure    69    2208    1.0
start      drive      68    2176    1.0
           measure    68    2176    1.0

85 DS3 runs have frames (2026-05-21 to 2026-09-14), and 69 of them come before the last DS2 snapshot (2026-08-18 09:13). Every drive and measure frame in those 69 equals DS2 as of `created_at`, and as of the benchmarking start for the 68 with a linked start. So the frames copy the hardware model, as datasheet 3 says.

## Conclusion

| Result | Status |
|---|---|
| From 2025 every run links to DS1 and DS2, and 92.4% of benchmarking runs link to DS3. From 2024-09-18 only DS1 and DS3 link, and before that nothing does | Confirmed |
| A run's old values are the DS2 snapshot at or before its start, and its written values the first one after. Changed values reach DS2, and tracked runs own over 99% of DS2 changes | Confirmed |
| The DS3 link pairs the right files: served $R^2$ matches the linked file every time and the file before never | Confirmed |
| Values agree along the chain, from `output_info` to DS2, the DS1 Ramsey fits and the DS3 frames. Ramsey `new` is the applied f01 only when the run has no FineFrequency step for the qubit | Confirmed |
| A warning isn't a rejection, and neither is a missing or unchanged value on its own | Confirmed |
| 1q rows in the run with no step id or warning aren't attempts: almost all are on `freq_shift` qubits or are Rabi-only amp rows | Confirmed (330 rows unexplained) |
| Ids from runs before the DS1 restore (2024-09-18) point at the wrong rows, so we filter on `valid`. That leaves about half of DS1 claimed, and none of the mixer sub-steps or readout types | Known limit |
| DS2 stops recording DC bias after 2026-05-25 22:01, so we take it from `output_info` after that | Known limit |
| 38 benchmarking files have no linked DS3 run that day, and six of them have over half their $R^2$ below 0.9 | Open |